### Step:1 Importing all the libraries 


In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

### Step 2: Loading the datasets 


In [18]:
df = pd.read_csv("dataset/earthquake.csv")

### Step 3: Look at the dataset first

In [19]:
print("Shape of dataset (rows, columns):", df.shape)
print("\nFirst 5 rows of dataset:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values in each column:")
print(df.isnull().sum())

Shape of dataset (rows, columns): (17100, 22)

First 5 rows of dataset:
                       time  latitude  longitude    depth  mag magType    nst  \
0  2026-05-30T21:15:59.807Z   38.3564    73.8219  131.794  4.9      mb   80.0   
1  2026-05-29T12:22:54.128Z   23.4621    93.7220   57.374  4.3      mb   32.0   
2  2026-05-28T02:37:50.460Z   33.1393    96.1517   10.000  4.7      mb  111.0   
3  2026-05-26T16:30:57.714Z   23.1849    94.5425  102.712  4.3      mb   21.0   
4  2026-05-26T14:07:51.354Z   23.7982    94.8304  112.776  4.5      mb   36.0   

     gap   dmin   rms  ...                   updated  \
0   78.0  1.052  0.94  ...  2026-05-30T21:35:21.040Z   
1  126.0  2.503  0.68  ...  2026-05-29T12:58:49.040Z   
2   47.0  5.492  0.73  ...  2026-05-28T05:26:32.040Z   
3  158.0  2.000  0.43  ...  2026-05-27T14:59:41.040Z   
4   68.0  2.581  0.40  ...  2026-05-26T17:07:05.040Z   

                                   place        type horizontalError  \
0       24 km NNW of Murghob, Ta

### STEP 4: DATA PREPROCESSING
#### latitude, longitude and depth (these describe WHERE the earthquake happened) we also keep magnitude to study the clusters later

In [20]:
columns_needed = ["latitude", "longitude", "depth", "mag"]
df_cluster = df[columns_needed].copy()

### Remove rows that have missing values (simple way to handle missing data)

In [21]:
print("\nRows before removing missing values:", df_cluster.shape[0])
df_cluster = df_cluster.dropna()
print("Rows after removing missing values:", df_cluster.shape[0])


Rows before removing missing values: 17100
Rows after removing missing values: 17100


### Remove duplicate rows if any

In [22]:
df_cluster = df_cluster.drop_duplicates()
print("Rows after removing duplicates:", df_cluster.shape[0])

Rows after removing duplicates: 17100


### STEP 5: FEATURE ENGINEERING

In [23]:
features = df_cluster[["latitude", "longitude", "depth"]]

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

print("\nFeatures before scaling (first 5 rows):")
print(features.head())

print("\nFeatures after scaling (first 5 rows):")
print(features_scaled[:5])


Features before scaling (first 5 rows):
   latitude  longitude    depth
0   38.3564    73.8219  131.794
1   23.4621    93.7220   57.374
2   33.1393    96.1517   10.000
3   23.1849    94.5425  102.712
4   23.7982    94.8304  112.776

Features after scaling (first 5 rows):
[[ 0.98982295 -0.50975535  1.12802173]
 [-1.59956116  1.34002445 -0.05821969]
 [ 0.08282659  1.56587307 -0.81335272]
 [-1.64775257  1.41629263  0.66445988]
 [-1.54112995  1.44305388  0.82487823]]


### STEP 6: FIND THE BEST NUMBER OF CLUSTERS (K)
###  We use the Elbow Method to choose a good value of K
### We try K = 2 to 10 and check the "inertia" (how tight the clusters are)


In [24]:
inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_test.fit(features_scaled)
    inertia_values.append(kmeans_test.inertia_)

print("\nInertia values for K = 2 to 10:")
for k, inertia in zip(k_range, inertia_values):
    print(f"K = {k} -> Inertia = {inertia:.2f}")


Inertia values for K = 2 to 10:
K = 2 -> Inertia = 27049.41
K = 3 -> Inertia = 15933.76
K = 4 -> Inertia = 11978.06
K = 5 -> Inertia = 9047.21
K = 6 -> Inertia = 7121.30
K = 7 -> Inertia = 6114.72
K = 8 -> Inertia = 5319.03
K = 9 -> Inertia = 4723.45
K = 10 -> Inertia = 4194.94


### We also check the Silhouette Score for a few K values
### Silhouette score tells us how well separated the clusters are (closer to 1 is better)

In [25]:
print("\nSilhouette scores for K = 2 to 8:")
silhouette_scores = []
for k in range(2, 9):
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_test = kmeans_test.fit_predict(features_scaled)
    score = silhouette_score(features_scaled, labels_test)
    silhouette_scores.append(score)
    print(f"K = {k} -> Silhouette Score = {score:.4f}")


Silhouette scores for K = 2 to 8:
K = 2 -> Silhouette Score = 0.4491
K = 3 -> Silhouette Score = 0.4710
K = 4 -> Silhouette Score = 0.4421
K = 5 -> Silhouette Score = 0.4620
K = 6 -> Silhouette Score = 0.4751
K = 7 -> Silhouette Score = 0.4700
K = 8 -> Silhouette Score = 0.4667


### Based on the elbow method and silhouette score, we choose our final K
### (After checking both results, K = 5 was selected for this project)

In [26]:
best_k = 5

### STEP 7: TRAIN THE FINAL K-MEANS MODEL

In [27]:
kmeans_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster["cluster"] = kmeans_model.fit_predict(features_scaled)

print(f"\nFinal model trained with K = {best_k}")
print("\nNumber of earthquakes in each cluster:")
print(df_cluster["cluster"].value_counts())


Final model trained with K = 5

Number of earthquakes in each cluster:
cluster
0    5517
4    3614
3    3602
1    2928
2    1439
Name: count, dtype: int64


### STEP 8: EVALUATE THE MODEL

In [28]:
final_silhouette = silhouette_score(features_scaled, df_cluster["cluster"])
print(f"\nFinal Silhouette Score for K = {best_k}: {final_silhouette:.4f}")

# Show the center of each cluster (in original scale, not scaled values)
cluster_centers_scaled = kmeans_model.cluster_centers_
cluster_centers_original = scaler.inverse_transform(cluster_centers_scaled)

print("\nCluster centers (latitude, longitude, depth):")
centers_df = pd.DataFrame(cluster_centers_original, columns=["latitude", "longitude", "depth"])
print(centers_df)


Final Silhouette Score for K = 5: 0.4620

Cluster centers (latitude, longitude, depth):
    latitude  longitude       depth
0  36.948599  72.806412   34.331835
1  23.151581  94.966067   50.000878
2  27.739034  67.331068   22.356468
3  31.699581  89.338660   21.548243
4  36.749892  71.316437  165.480466


### STEP 9: ANALYZE EACH CLUSTER (FINDINGS)

In [29]:
print("\nAverage magnitude and depth per cluster:")
cluster_summary = df_cluster.groupby("cluster")[["mag", "depth"]].mean()
print(cluster_summary)

print("\nMax magnitude found in each cluster:")
cluster_max_mag = df_cluster.groupby("cluster")["mag"].max()
print(cluster_max_mag)


Average magnitude and depth per cluster:
              mag       depth
cluster                      
0        4.586523   34.333630
1        4.609747   49.981410
2        4.633516   22.324619
3        4.652252   21.548270
4        4.516630  165.480466

Max magnitude found in each cluster:
cluster
0    7.6
1    7.7
2    7.7
3    7.8
4    7.5
Name: mag, dtype: float64


### STEP 10: SAVE THE RESULTS

In [30]:
df_cluster.to_csv("dataset/results/earthquake_with_clusters.csv", index=False)
print("\nClustered data saved to earthquake_with_clusters.csv")

print("\n=== K-MEANS CLUSTERING COMPLETE ===")


Clustered data saved to earthquake_with_clusters.csv

=== K-MEANS CLUSTERING COMPLETE ===
